# ShadowCrafter-9B V2 — VS Code + Google Colab

이 노트북은 공식 VS Code 확장 `google.colab`에서 실행하는 V2 학습 진입점입니다. `Select Kernel → Colab → Auto Connect`로 런타임을 연결하세요.

- 기반 모델: `ornith-ai/Ornith-1.5-9B` 고정 revision
- 학습 자료: v1 28,140건 + NIST Juliet C/C++ 64,099건 = 92,239건
- 평가 자료: CTIBench 5,533건은 오염 검사에만 사용하고 학습하지 않음
- 권장 GPU: A100 40GB 이상. GPU가 보장되지 않는 무료 Colab에는 권장하지 않음
- 체크포인트: private Google Drive에 SHA-256 완전성 마커와 함께 저장

Drive 체크포인트는 무결성 manifest이지 전자서명이 아닙니다. 본인만 접근 가능한 Drive 폴더를 사용하세요. 토큰이나 개인 키를 셀에 입력하거나 저장하지 마세요.

## 0. 최초 1회 입력 파일 업로드

Google Drive의 `MyDrive/ShadowCrafterV2/inputs/` 아래에 다음 두 파일을 업로드하세요. 로컬 파일은 Git에서 제외된 기존 mirror에 있습니다.

| Drive 대상 | 로컬 원본 | SHA-256 |
|---|---|---|
| `v1/train.jsonl` | `local_mirror/remote-project/data/processed/security-expanded-20260901-v8-blackbox-train-only/train.jsonl` | `8b0be9434be7452bf8129650eec485a00d2ce3efabeb725dc2f81908e18b7c7f` |
| `ctibench/cases.jsonl` | `local_mirror/remote-project/artifacts/evaluations/ctibench-9237e163/cases.jsonl` | `2455b46b4851ed998ce3094ba7d9f796365bd0d71ce51264ff665f1c5203b423` |

Private GitHub clone에는 Colab Secrets의 `GITHUB_TOKEN`을 사용합니다. Fine-grained read-only token으로 `Odytssey/ShadowCrafter`만 허용하세요.

In [ ]:
# 1. Colab GPU 및 Drive 연결
import json, os, shutil, subprocess, sys
from pathlib import Path

if not Path('/content').is_dir():
    raise RuntimeError('이 노트북은 Google Colab 런타임에서 실행해야 합니다.')
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
    check=True, capture_output=True, text=True,
).stdout.strip()
print('GPU:', gpu)
memory_mib = int(gpu.rsplit(',', 1)[1].strip())
if memory_mib < 38_000:
    raise RuntimeError('공식 V2 설정은 GPU VRAM 40GB 이상이 필요합니다. A100급 런타임을 선택하세요.')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/ShadowCrafterV2')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

In [ ]:
# 2. Private GitHub를 토큰 노출 없이 clone하고 exact revision으로 detach
import base64
from google.colab import userdata

REPO_DIR = Path('/content/ShadowCrafter')
REPO_URL = 'https://github.com/Odytssey/ShadowCrafter.git'
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Colab Secrets에 read-only GITHUB_TOKEN을 추가하세요.')
credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
git_env = os.environ.copy()
git_env.update({
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.extraHeader',
    'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credential}',
    'GIT_TERMINAL_PROMPT': '0',
})
if REPO_DIR.exists():
    raise RuntimeError(f'기존 경로를 자동 삭제하지 않습니다: {REPO_DIR}. 새 런타임을 사용하세요.')
subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_DIR)], env=git_env, check=True)
SOURCE_PIN = DRIVE_ROOT / 'SOURCE_REVISION'
if SOURCE_PIN.exists():
    if SOURCE_PIN.is_symlink():
        raise RuntimeError('SOURCE_REVISION pin은 symlink일 수 없습니다.')
    SOURCE_REVISION = SOURCE_PIN.read_text().strip()
else:
    SOURCE_REVISION = subprocess.run(
        ['git', '-C', str(REPO_DIR), 'rev-parse', 'origin/main'],
        check=True, capture_output=True, text=True, env=git_env,
    ).stdout.strip()
    with SOURCE_PIN.open('x', encoding='utf-8') as handle:
        handle.write(SOURCE_REVISION + '\n')
if len(SOURCE_REVISION) != 40 or any(ch not in '0123456789abcdef' for ch in SOURCE_REVISION):
    raise RuntimeError('Drive SOURCE_REVISION pin 형식이 잘못됐습니다.')
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'checkout', '--detach', SOURCE_REVISION], env=git_env, check=True
)
del github_token, credential, git_env
print('Pinned source revision:', SOURCE_REVISION)

In [ ]:
# 3. 감사된 학습 runtime 설치 및 버전 확인
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(REPO_DIR / 'requirements/train-hf.lock.txt')],
    check=True,
)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(REPO_DIR)], check=True)
from importlib.metadata import version
EXPECTED = {
    'accelerate': '1.14.0', 'bitsandbytes': '0.50.2', 'datasets': '4.8.5',
    'huggingface-hub': '1.29.0', 'peft': '0.19.1', 'safetensors': '0.8.0',
    'torch': '2.10.0', 'transformers': '5.12.1', 'trl': '0.29.1',
}
observed = {name: version(name) for name in EXPECTED}
mismatch = {name: {'expected': EXPECTED[name], 'actual': value} for name, value in observed.items() if value != EXPECTED[name]}
if mismatch:
    raise RuntimeError(f'패키지 버전 불일치입니다. Kernel을 재시작하고 1번 셀부터 다시 실행하세요: {mismatch}')
print(json.dumps(observed, indent=2, sort_keys=True))

In [ ]:
# 4. 고정 Ornith base model을 내려받고 18개 파일의 크기/SHA-256 검증
import hashlib
from huggingface_hub import snapshot_download

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

BASE_REVISION = '489cb97981b8654bcfcf30ce1f94ed1b62e07b53'
BASE_MANIFEST = REPO_DIR / 'artifacts/manifests/ornith-1.5-9b.json'
BASE_MANIFEST_SHA256 = '9a8c8c0c909311654a8ced2181b838cfc6d1db08d82f81b841cefa9030178f94'
if sha256_file(BASE_MANIFEST) != BASE_MANIFEST_SHA256:
    raise RuntimeError('base model manifest pin이 다릅니다.')
base_manifest = json.loads(BASE_MANIFEST.read_text())
allowed = [entry['path'] for entry in base_manifest['files']]
snapshot = Path(snapshot_download('ornith-ai/Ornith-1.5-9B', revision=BASE_REVISION, allow_patterns=allowed))
BASE_MODEL = Path('/content/base-model/Ornith-1.5-9B')
BASE_MODEL.mkdir(parents=True, exist_ok=True)
for entry in base_manifest['files']:
    source = snapshot / entry['path']
    target = BASE_MODEL / entry['path']
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        shutil.copyfile(source, target)
    if target.is_symlink() or target.stat().st_size != entry['size'] or sha256_file(target) != entry['sha256']:
        raise RuntimeError(f'base model 파일 검증 실패: {entry["path"]}')
print('Verified base model bytes:', sum(item['size'] for item in base_manifest['files']))

In [ ]:
# 5. V2 92,239건 corpus 생성/검증 후 Drive에 immutable cache 저장
import urllib.request, uuid
from datetime import UTC, datetime
from shadowcrafter.data.adapters import canonicalize_nist_juliet
from shadowcrafter.data.ctibench import find_ctibench_training_contamination, load_ctibench_eval_cases
from shadowcrafter.data.prepare import SplitMode, prepare_jsonl_many
from shadowcrafter.schemas import SecurityRecord

V1_SHA256 = '8b0be9434be7452bf8129650eec485a00d2ce3efabeb725dc2f81908e18b7c7f'
CTI_SHA256 = '2455b46b4851ed998ce3094ba7d9f796365bd0d71ce51264ff665f1c5203b423'
JULIET_SHA256 = 'ada9d7e1c323d283446df3f55bdee0d00bda1fed786785fe98764d58688f38eb'
V1_DRIVE = DRIVE_ROOT / 'inputs/v1/train.jsonl'
CTI_DRIVE = DRIVE_ROOT / 'inputs/ctibench/cases.jsonl'
for path, digest, label in [(V1_DRIVE, V1_SHA256, 'v1 train'), (CTI_DRIVE, CTI_SHA256, 'CTIBench')]:
    if not path.is_file():
        raise FileNotFoundError(f'{label} 입력을 Drive에 먼저 업로드하세요: {path}')
    if sha256_file(path) != digest:
        raise RuntimeError(f'{label} 입력 SHA-256이 다릅니다: {path}')

DATA_CACHE = DRIVE_ROOT / 'datasets' / f'v2.0-92239-{SOURCE_REVISION[:12]}'
LOCAL_DATA = Path('/content/shadowcrafter-v2-data')
if DATA_CACHE.is_dir() and (DATA_CACHE / 'READY.json').is_file():
    ready = json.loads((DATA_CACHE / 'READY.json').read_text())
    if ready.get('record_count') != 92_239:
        raise RuntimeError('Drive dataset cache record count가 다릅니다.')
    if not LOCAL_DATA.exists():
        shutil.copytree(DATA_CACHE / 'processed', LOCAL_DATA)
    if sha256_file(LOCAL_DATA / 'train.jsonl') != ready['train_sha256']:
        raise RuntimeError('Drive dataset cache SHA-256 검증 실패')
    if sha256_file(LOCAL_DATA / 'manifest.json') != ready['manifest_sha256']:
        raise RuntimeError('Drive dataset manifest SHA-256 검증 실패')
else:
    if LOCAL_DATA.exists():
        raise RuntimeError(f'검증되지 않은 기존 local dataset 경로가 있습니다: {LOCAL_DATA}')
    WORK = Path('/content') / f'shadowcrafter-v2-work-{uuid.uuid4().hex}'
    WORK.mkdir(parents=True, exist_ok=False)
    v1_local = WORK / 'v1-train.jsonl'
    cti_local = WORK / 'ctibench-cases.jsonl'
    shutil.copyfile(V1_DRIVE, v1_local)
    shutil.copyfile(CTI_DRIVE, cti_local)
    juliet_zip = WORK / 'juliet-cpp-1.3.zip'
    urllib.request.urlretrieve(
        'https://samate.nist.gov/SARD/downloads/test-suites/2017-10-01-juliet-test-suite-for-c-cplusplus-v1-3.zip',
        juliet_zip,
    )
    if sha256_file(juliet_zip) != JULIET_SHA256:
        raise RuntimeError('NIST Juliet 공식 ZIP SHA-256 검증 실패')
    juliet_jsonl = WORK / 'juliet.jsonl'
    juliet_manifest = canonicalize_nist_juliet(
        juliet_zip, juliet_jsonl, upstream_revision='nist-sard-suite-112-juliet-cpp-1.3',
        retrieved_at=datetime.now(UTC), registry_path=REPO_DIR / 'configs/data/sources.yaml',
    )
    cases = load_ctibench_eval_cases(cti_local)
    juliet_records = [SecurityRecord.model_validate_json(line) for line in juliet_jsonl.read_text().splitlines() if line]
    overlap = find_ctibench_training_contamination(juliet_records, cases)
    if overlap:
        raise RuntimeError(f'Juliet/CTIBench 오염 발견: {len(overlap)}')
    prepared = prepare_jsonl_many(
        [v1_local, juliet_jsonl], LOCAL_DATA, registry_path=REPO_DIR / 'configs/data/sources.yaml',
        split_mode=SplitMode.TRAIN_ONLY,
    )
    if prepared['record_count'] != 92_239 or prepared['split_counts']['train'] != 92_239:
        raise RuntimeError(f'V2 record count 불일치: {prepared["split_counts"]}')
    if prepared['exact_duplicate_count'] or prepared['normalized_duplicate_count']:
        raise RuntimeError('V2 corpus에 duplicate가 남았습니다.')
    staging = DATA_CACHE.parent / f'.{DATA_CACHE.name}.staging-{uuid.uuid4().hex}'
    staging.mkdir(parents=True, exist_ok=False)
    shutil.copytree(LOCAL_DATA, staging / 'processed')
    ready = {
        'schema_version': 1, 'record_count': 92_239,
        'train_sha256': sha256_file(LOCAL_DATA / 'train.jsonl'),
        'manifest_sha256': sha256_file(LOCAL_DATA / 'manifest.json'),
        'dataset_sha256': prepared['dataset_sha256'], 'ctibench_overlap': 0,
    }
    (staging / 'READY.json').write_text(json.dumps(ready, indent=2, sort_keys=True) + '\n')
    DATA_CACHE.parent.mkdir(parents=True, exist_ok=True)
    staging.rename(DATA_CACHE)
print(json.dumps(ready, indent=2, sort_keys=True))

In [ ]:
# 6. SHA-256 완전성 checkpoint에서 자동 재개하며 V2 QLoRA 학습
from shadowcrafter.data.manifest import sha256_file as project_sha256_file
from shadowcrafter.data.registry import load_registry
from shadowcrafter.training.colab import train_resumable_colab
from shadowcrafter.training.training_safety import TrainingPins

CONFIG = REPO_DIR / 'configs/models/shadowcrafter-9b.yaml'
REGISTRY = REPO_DIR / 'configs/data/sources.yaml'
TRAIN = LOCAL_DATA / 'train.jsonl'
DATA_MANIFEST = LOCAL_DATA / 'manifest.json'
dataset_manifest = json.loads(DATA_MANIFEST.read_text())
identity = f'{SOURCE_REVISION[:12]}-{dataset_manifest["dataset_sha256"][:12]}'
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints' / identity
FINAL_DIR = DRIVE_ROOT / 'candidates' / f'v2.0-{identity}'
pins = TrainingPins(
    config_sha256=project_sha256_file(CONFIG),
    train_sha256=project_sha256_file(TRAIN),
    validation_sha256=None,
    dataset_manifest_sha256=project_sha256_file(DATA_MANIFEST),
    registry_sha256=load_registry(REGISTRY).canonical_sha256(),
    git_revision=SOURCE_REVISION,
)
if FINAL_DIR.exists():
    print('이미 완료된 immutable candidate가 있습니다:', FINAL_DIR)
    run_manifest = json.loads((FINAL_DIR / 'run-manifest.json').read_text())
    adapter_path = FINAL_DIR / 'adapter/adapter_model.safetensors'
    if sha256_file(adapter_path) != run_manifest['adapter']['adapter_weights_sha256']:
        raise RuntimeError('기존 final candidate adapter SHA-256 검증 실패')
else:
    run_manifest = train_resumable_colab(
        config_path=CONFIG, train_path=TRAIN, dataset_manifest_path=DATA_MANIFEST,
        registry_path=REGISTRY, base_model_path=BASE_MODEL,
        base_model_manifest_path=BASE_MANIFEST,
        base_model_manifest_sha256=BASE_MANIFEST_SHA256,
        checkpoint_root=CHECKPOINT_ROOT, final_dir=FINAL_DIR, pins=pins,
        save_steps=100, save_total_limit=3,
    )
print(json.dumps({
    'candidate': str(FINAL_DIR),
    'global_step': run_manifest['training_observation']['global_step'],
    'train_loss': run_manifest['training_observation']['train_loss'],
    'adapter_sha256': run_manifest['adapter']['adapter_weights_sha256'],
}, indent=2, sort_keys=True))

In [ ]:
# 7. 현재 체크포인트/완료 상태 확인 — 언제든 다시 실행 가능
subprocess.run(['nvidia-smi'], check=False)
markers = sorted(CHECKPOINT_ROOT.glob('checkpoint-*/.shadowcrafter-complete.json'))
print('Complete checkpoints:', len(markers))
for marker in markers[-3:]:
    payload = json.loads(marker.read_text())
    print(marker.parent.name, 'files=', len(payload['files']))
print('Final candidate exists:', FINAL_DIR.is_dir())

## 학습 완료 후

`FINAL_DIR`에 검증된 LoRA adapter와 `run-manifest.json`이 생성됩니다. 이 노트북은 학습 중 Hugging Face에 자동 업로드하지 않습니다. 다음 단계는 candidate를 원격 검증 경로로 동기화하고 CTIBench 5,533건을 새로 평가한 뒤, 실제 정확도를 기록해 public `v2.0`으로 게시하는 것입니다.